In [26]:
from transformers import AutoProcessor, AutoModelForCausalLM

model_id = 'google/functiongemma-270m-it'
processor = AutoProcessor.from_pretrained(model_id, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 358.56it/s]


In [27]:
# Single Tool Calling example

weather_function_schema = {
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Gets the current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco",
                },
            },
            "required": ["location"],
        },
    }
}

message = [
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    {
        "role": "user", 
        "content": "What's the temperature in London?"
    },
]

inputs = processor.apply_chat_template(
    message, 
    tools=[weather_function_schema], 
    add_generation_prompt=False, 
    return_dict=True,
    return_tensors="pt"
)

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(output)

<start_function_call>call:get_current_temperature{location:<escape>London<escape>}<end_function_call>


In [28]:
# Parallel Tool Calling example

weather_function_schema = {
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Gets the current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco",
                },
            },
            "required": ["location"],
        },
    }
}

stock_price_function_schema = {
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "Gets the stock price for a given stock name",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "The name of the stock to fetch the price for",
                },
            },
            "required": ["name"],
        },
    }
}

message = [
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    {
        "role": "user", 
        "content": "What's the temperature in London?"
    },
    {
        "role": "user", 
        "content": "What's the stock price of Google?"
    }
]

inputs = processor.apply_chat_template(
    message, 
    tools=[weather_function_schema, stock_price_function_schema], 
    add_generation_prompt=False, 
    return_dict=True,
    return_tensors="pt"
)

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(output)

<start_function_call>call:get_current_temperature{location:<escape>London<escape>}<end_function_call><start_function_call>call:get_stock_price{name:<escape>Google<escape>}<end_function_call>
